# Mesenchyme annotation

Takes the mesenchymal compartment out of `compartments.ipynb` and assigns
`cell_type` labels.

Two clustering passes, same shape as `epithelial.ipynb`:

1. **Cluster, inspect, remove junk.**
2. **Re-cluster what remains, then annotate.** Removing cells changes every
   neighbourhood, so the graph and UMAP are rebuilt before the clustering the
   labels are assigned to.

Set `DATA` in `config.py` before running.

In [ ]:
import pandas as pd
import scanpy as sc

from config import DATA

SEED = 0

## 1. Load the mesenchymal compartment

Written by `compartments.ipynb`.

In [ ]:
adata = sc.read_h5ad(DATA / 'mesenchyme.h5ad')
print(adata.shape)

## 2. First pass - cluster to find the junk

> **The resolution below is not a principled value.** It is tuned by eye, and
> the only thing it has to achieve is putting the cells you want to remove into
> a cluster of their own - not split across several, and not mixed in with a
> real population. Raise it until the junk separates cleanly, and no further.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# tune this by eye until the junk sits in its own cluster - see above
sc.tl.leiden(adata, resolution=0.5, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False)

Check the clusters before deciding what to drop. A cluster is junk when it has
no distinguishing marker of its own, or when it co-expresses markers of another
compartment - residual `Pecam1` endothelium surviving the split is the usual
offender, and this is where it gets caught.

In [ ]:
sc.pl.umap(
    adata,
    color=['Pecam1', 'Ptprc', 'Epcam', 'n_genes_by_counts'],
    frameon=False
)

## 3. Drop the junk clusters

> **Cluster numbers here belong to the first pass only**, at whatever
> resolution was used above. Re-running this notebook renumbers them. Read the
> plots and rebuild this list.

In [ ]:
JUNK = ['10']   # no distinguishing marker

n_before = adata.n_obs
adata = adata[~adata.obs['leiden'].isin(JUNK)].copy()

print(f'{n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} dropped)')

## 4. Second pass - re-cluster what is left

The resolution is raised by eye until every population below separates. The
binding constraint is FDC: they sit inside the adventitial fibroblasts, which is
where they belong anatomically - the perivascular niche where tertiary lymphoid
structures form - and a resolution coarse enough to leave the fibroblast
subsets intact will leave FDC buried in with them. Raise it until a small
`Cxcl13`-high cluster appears, and stop there.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

sc.tl.leiden(adata, resolution=0.5, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False)

## 5. Markers

Each mesenchymal population is called from its own marker set rather than a
single gene - fibroblast subsets differ by degree more than by presence or
absence, so a panel separates them where one gene does not.

| Population | Markers |
| --- | --- |
| Adventitial Fibroblast | `Dcn`, `Igfbp4`, `Gsn`, `Clec3b`, `Col14a1`, `Serpinf1`, `Ly6a` |
| Alveolar Fibroblast | `Inmt`, `Limch1`, `Macf1`, `Gsn`, `Fmo2`, `Pcolce2`, `C7`, `Ddit3` |
| Cycling Fibroblast | `Stmn1`, `Hmgb2`, `Ran`, `Lgals1`, `Cks1b`, `Mki67` |
| Fibrotic Fibroblast | `Col1a1`, `Serpine2`, `Nrep`, `Tnc`, `Fn1`, `Col5a2` |
| Inflammatory Fibroblast | `Tmem176a`, `Fn1`, `Hp`, `Tmem176b`, `Serpina3n`, `Gyg`, `Limch1` |
| Mesothelial Cells | `C3`, `Upk3b`, `Gpm6a`, `Clu`, `Ezr`, `Dcn`, `Msln` |
| Peribronchial Fibroblast | `Cd9`, `P2ry14`, `Mustn1`, `Aspn`, `Igfbp5`, `Lum`, `Sema3c` |
| Pericyte | `Postn`, `Notch3`, `Pdgfrb`, `Ndufa4l2`, `Crim1`, `Itga1`, `Ebf1` |
| Smooth Muscle | `Acta2`, `Tagln`, `Myh11`, `Myl9`, `Tpm2`, `Map1b`, `Lmod1` |
| FDC | `Cxcl13`, `Tnfsf13b` |

Some genes appear in more than one panel - `Gsn` in both adventitial and
alveolar, `Fn1` in both fibrotic and inflammatory, `Dcn` in both adventitial and
mesothelial. Those are read together with the rest of their panel, not alone.

In [ ]:
# adventitial
sc.pl.umap(adata, color=['Dcn', 'Igfbp4', 'Gsn', 'Clec3b', 'Col14a1',
                         'Serpinf1', 'Ly6a'], frameon=False)

In [ ]:
# alveolar
sc.pl.umap(adata, color=['Inmt', 'Limch1', 'Macf1', 'Gsn', 'Fmo2',
                         'Pcolce2', 'C7', 'Ddit3'], frameon=False)

In [ ]:
# cycling
sc.pl.umap(adata, color=['Stmn1', 'Hmgb2', 'Ran', 'Lgals1', 'Cks1b',
                         'Mki67'], frameon=False)

In [ ]:
# fibrotic
sc.pl.umap(adata, color=['Col1a1', 'Serpine2', 'Nrep', 'Tnc', 'Fn1',
                         'Col5a2'], frameon=False)

In [ ]:
# inflammatory
sc.pl.umap(adata, color=['Tmem176a', 'Fn1', 'Hp', 'Tmem176b', 'Serpina3n',
                         'Gyg', 'Limch1'], frameon=False)

In [ ]:
# mesothelial
sc.pl.umap(adata, color=['C3', 'Upk3b', 'Gpm6a', 'Clu', 'Ezr', 'Dcn',
                         'Msln'], frameon=False)

In [ ]:
# peribronchial fibroblast
sc.pl.umap(adata, color=['Cd9', 'P2ry14', 'Mustn1', 'Aspn', 'Igfbp5',
                         'Lum', 'Sema3c'], frameon=False)

In [ ]:
# pericyte
sc.pl.umap(adata, color=['Postn', 'Notch3', 'Pdgfrb', 'Ndufa4l2', 'Crim1',
                         'Itga1', 'Ebf1'], frameon=False)

In [ ]:
# smooth muscle
sc.pl.umap(adata, color=['Acta2', 'Tagln', 'Myh11', 'Myl9', 'Tpm2',
                         'Map1b', 'Lmod1'], frameon=False)

In [ ]:
# FDC
sc.pl.umap(adata, color=['Cxcl13', 'Tnfsf13b'], frameon=False)

Ranked marker genes per cluster, as a cross-check on the panels above. Clusters
whose top genes do not match the call below are worth revisiting.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# one column per cluster, top 100 genes down the rows - this is what the calls
# below were read from
top_genes = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(100)
top_genes.head(30)

## 6. Assign cell types

Read off the marker panels and the ranked genes above. Clusters 1 and 6 are
both alveolar fibroblasts, and 2 and 7 both inflammatory - Leiden splits these
further than the biology justifies, and merging them here is deliberate.

> These numbers belong to the second clustering pass in this notebook. They are
> a record of the run that produced the deposited object, not something to
> trust after a re-run.

**The entries below are an example, not the full mapping.** At the resolution
this ran at there were closer to twenty clusters, every one of them written out
by hand. Only the shape is meant to carry over: one entry per cluster, and many
clusters landing on the same label. A dict with three clusters all called
`Alveolar Fibroblast` is normal - Leiden splits the large populations further
than the biology justifies, and merging them here is the point.

Rebuild it for whatever clustering you end up with.

In [ ]:
# example only - the run this documents had roughly twenty clusters
CELL_TYPES = {
    '0':  'Fibrotic Fibroblast',
    '1':  'Alveolar Fibroblast',
    '2':  'Inflammatory Fibroblast',
    '3':  'Adventitial Fibroblast',
    '4':  'Pericyte',
    '5':  'Smooth Muscle',
    '6':  'Alveolar Fibroblast',
    '7':  'Inflammatory Fibroblast',
    '8':  'Peribronchial Fibroblast',
    '9':  'Mesothelial Cells',
    '10': 'Cycling Fibroblast',
    '11': 'FDC',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(CELL_TYPES).astype('category')
print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='cell_type', frameon=False)

Composition per condition - the reason the annotation exists.

In [ ]:
sc.pl.umap(adata, color=['type', 'age'], frameon=False)

## 7. Save

In [ ]:
adata.write_h5ad(DATA / 'mesenchyme_annotated.h5ad')
print(f'{adata.n_obs:,} cells, {adata.obs["cell_type"].nunique()} cell types')